# Gemini-3-Flash-Preview (dual-bfws-ffparser)

In [ ]:
# Cell 1: Setup imports, paths, and Gemini key (new google.genai SDK)
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "only_pddl_instruction.txt"
DOMAIN_PDDL_PATH = DATA_DIR / "pddl" / "domain.pddl"
DESC_DIR = DATA_DIR / "descriptions"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl" / "gemini_3_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate")

# Load .env and configure Gemini client
load_dotenv(PROJECT_ROOT / ".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
  raise ValueError("GOOGLE_API_KEY not found in .env")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Gemini 3 Flash model code (preview)
MODEL_NAME = "gemini-3-flash-preview"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("Domain PDDL exists:", DOMAIN_PDDL_PATH.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

In [ ]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:03d}" for i in range(101, 201)]  # p101..p200

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

In [ ]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [ ]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Create a syntactically correct and semantically consistent problem file "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the final problem PDDL.\n"
      "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <problem_file>...</problem_file>."
  )

In [ ]:
# Cell 5: Call Gemini for one prompt
def call_gemini(prompt: str, temperature: float = 0.4, max_output_tokens: int = 50000):
  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config={
          "temperature": temperature,
          "max_output_tokens": max_output_tokens,
      },
  )
  return (response.text or "").strip()

In [ ]:
# Cell 6: Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [ ]:
# Cell 7: Solve one generated problem via planning.domains API (online solver)
import requests
import re

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic:
  # 1) explicit plan text exists
  # 2) solver stdout says a plan was found (covers cost-0 empty-plan case)
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          # cost-0 case: keep empty plan file so VAL can still run
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,  # can be empty for cost-0
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [ ]:
# Cell 7b: Solve one generated problem via planning.domains API (EC2 self-hosted endpoint, resilient polling)
import requests
import re
from requests.exceptions import ReadTimeout, ConnectionError, ChunkedEncodingError

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 1200):
  solver_name = "dual-bfws-ffparser"
  base_url = "http://100.51.78.30:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=(10, 120),
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=(5, 180))
          poll.raise_for_status()
          res = poll.json()
      except (ReadTimeout, ConnectionError, ChunkedEncodingError, ConnectionResetError):
          print("[solver-online] transient poll error; retrying...")
          time.sleep(1.0)
          continue
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic:
  # 1) explicit plan text exists
  # 2) solver stdout says a plan was found (covers cost-0 empty-plan case)
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          # cost-0 case: keep empty plan file so VAL can still run
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,  # can be empty for cost-0
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [ ]:
# Cell 7c: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [ ]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.run(
      cmd,
      capture_output=True,
      text=True,
  )

  stdout = proc.stdout or ""
  stderr = proc.stderr or ""
  merged = (stdout + ("\n" + stderr if stderr else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [ ]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, problem_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated problem PDDL
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return cand_dir

In [ ]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)

  problem_pddl_text = extract_problem_pddl(raw_text)

  # If model fails to return a tagged problem file, record and stop early
  if not problem_pddl_text.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <problem_file>...</problem_file> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generated problem first so solver can consume it
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)
  gen_problem_path = problem_dir / "problem.pddl"
  gen_problem_path.write_text(problem_pddl_text)

  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=gen_problem_path,
      work_dir=problem_dir,
      timeout_sec=300,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      plan_path = problem_dir / "plan"
      if not plan_path.exists():
          # fallback safety if solver wrote plan text but not file
          plan_path = problem_dir / "plan.txt"
          plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(DOMAIN_PDDL_PATH, gen_problem_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts
  save_problem_outputs(problem_id, raw_text, problem_pddl_text, solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
  }

In [ ]:
# Cell 11: Generate only (no solver/VAL)
smoke_problem = "p60"

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
problem_pddl_text = extract_problem_pddl(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("saved:", cand_dir)

In [ ]:
# Cell 13: Batch generation-only (with tqdm), default p102..p110
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:03d}" for i in range(191, 201)]  # change this list for other batches

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
  try:
      prompt = build_prompt(pid)
      raw_text = call_gemini(prompt)
      problem_pddl_text = extract_problem_pddl(raw_text)

      cand_dir = OUTPUT_ROOT / pid / "cand_00"
      cand_dir.mkdir(parents=True, exist_ok=True)

      (cand_dir / "raw.txt").write_text(raw_text or "")
      (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
      (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

      ok = bool(problem_pddl_text.strip())
      gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})
      print(f"{pid}: generation {'ok' if ok else 'failed (no <problem_file>)'}")
  except Exception as e:
      gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
      print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r['generation_ok']))
print("failed:", sum(1 for r in gen_results if not r['generation_ok']))

# Parser for Problem Descriptions & Difference Checking

In [ ]:
# Cell 17: Parser section setup (independent)
from pathlib import Path
import re
import json

# Re-declare all paths needed for parser/diff section (no dependency on earlier cells)
PROJECT_ROOT = Path.cwd().parents[1]  # run notebook from formalizer-scaling/src/pddl_gen
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl" / "gemini_3_flash"

# Single-problem handle for parser + diff run
PARSER_PROBLEM = "p53"  # edit as needed

# Input files
NL_PROBLEM_PATH = DATA_DIR / "descriptions" / f"{PARSER_PROBLEM}_problem.txt"
LLM_PDDL_PATH = OUTPUT_ROOT / PARSER_PROBLEM / "cand_00" / "problem.pddl"

# Output folder/files
CAND_DIR = OUTPUT_ROOT / PARSER_PROBLEM / "cand_00"
CAND_DIR.mkdir(parents=True, exist_ok=True)

PARSED_PDDL_OUT = CAND_DIR / "parsed_problem.pddl"
DIFF_JSON_OUT = CAND_DIR / "parsed_vs_llm_diff.json"
DIFF_TXT_OUT = CAND_DIR / "parsed_vs_llm_diff.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem:", PARSER_PROBLEM)
print("NL exists:", NL_PROBLEM_PATH.exists())
print("LLM PDDL exists:", LLM_PDDL_PATH.exists())
print("Output dir:", CAND_DIR)

In [ ]:
# Cell 18: Helper 1 - normalize whitespace
def _norm_space(s: str) -> str:
  return re.sub(r"\s+", " ", s.strip())

In [ ]:
# Cell 19: Helper 2 - split NL section into ordered statements
def _split_statements(section_text: str):
  text = section_text.replace(", and ", ", ")
  parts = [_norm_space(p) for p in text.split(",")]
  return [p for p in parts if p]

In [ ]:
# Cell 20: Helper 3 - parse NL problem description into ordered facts
def parse_nl_problem_description(nl_text: str):
  text = _norm_space(nl_text)

  init_prefix = "As initial conditions I have that, "
  goal_prefix = "My goal is to have that "

  if init_prefix not in text or goal_prefix not in text:
      raise ValueError("Unexpected NL format: missing initial/goal prefixes.")

  init_start = text.index(init_prefix) + len(init_prefix)
  goal_start = text.index(goal_prefix)

  init_text = text[init_start:goal_start].strip()
  goal_text = text[goal_start + len(goal_prefix):].strip()

  init_statements = _split_statements(init_text)
  goal_statements = _split_statements(goal_text)

  init_facts = []
  goal_facts = []
  block_ids = set()

  re_clear = re.compile(r"^block (\d+) is clear\.$|^block (\d+) is clear$")
  re_on = re.compile(r"^block (\d+) is on top of block (\d+)\.$|^block (\d+) is on top of block (\d+)$")
  re_table = re.compile(r"^block (\d+) is on the table\.$|^block (\d+) is on the table$")
  re_hand = re.compile(r"^the hand is empty\.$|^the hand is empty$")

  def parse_stmt(stmt: str, in_goal: bool):
      st = stmt.strip()

      m = re_clear.match(st)
      if m:
          b = int(m.group(1) or m.group(2))
          block_ids.add(b)
          return ("clear", f"block{b}")

      m = re_on.match(st)
      if m:
          b1 = int(m.group(1) or m.group(3))
          b2 = int(m.group(2) or m.group(4))
          block_ids.update([b1, b2])
          return ("on", f"block{b1}", f"block{b2}")

      m = re_table.match(st)
      if m:
          b = int(m.group(1) or m.group(2))
          block_ids.add(b)
          return ("on-table", f"block{b}")

      m = re_hand.match(st)
      if m:
          if in_goal:
              raise ValueError("Unexpected 'the hand is empty' in goal section.")
          return ("arm-empty",)

      raise ValueError(f"Unrecognized statement: {st}")

  for s in init_statements:
      init_facts.append(parse_stmt(s, in_goal=False))

  for s in goal_statements:
      goal_facts.append(parse_stmt(s, in_goal=True))

  if not block_ids:
      raise ValueError("No block ids parsed from NL problem description.")

  n_blocks = max(block_ids)
  objects = [f"block{i}" for i in range(1, n_blocks + 1)]

  return {
      "objects": objects,
      "init_facts_ordered": init_facts,
      "goal_facts_ordered": goal_facts,
      "n_blocks": n_blocks,
  }

In [ ]:
# Cell 21: Helper 4 - render parsed facts into problem PDDL text
def render_problem_pddl(problem_id: str, parsed):
  lines = []
  lines.append(f"(define (problem blocksworld-{problem_id})")
  lines.append("  (:domain blocksworld)")
  lines.append(f"  (:objects {' '.join(parsed['objects'])} )")
  lines.append("  (:init ")

  for fact in parsed["init_facts_ordered"]:
      if fact[0] == "clear":
          lines.append(f"    (clear {fact[1]})")
      elif fact[0] == "on":
          lines.append(f"    (on {fact[1]} {fact[2]})")
      elif fact[0] == "on-table":
          lines.append(f"    (on-table {fact[1]})")
      elif fact[0] == "arm-empty":
          lines.append("    (arm-empty)")
      else:
          raise ValueError(f"Unknown init fact: {fact}")

  lines.append("  )")
  lines.append("  (:goal (and ")

  for fact in parsed["goal_facts_ordered"]:
      if fact[0] == "clear":
          lines.append(f"    (clear {fact[1]})")
      elif fact[0] == "on":
          lines.append(f"    (on {fact[1]} {fact[2]})")
      elif fact[0] == "on-table":
          lines.append(f"    (on-table {fact[1]})")
      elif fact[0] == "arm-empty":
          lines.append("    (arm-empty)")
      else:
          raise ValueError(f"Unknown goal fact: {fact}")

  lines.append("  ))")
  lines.append(")")
  return "\n".join(lines) + "\n"

In [ ]:
# Cell 22: Helper 5 - extract init/goal facts from a PDDL problem file (section-aware)
_fact_line_re = re.compile(r"\(\s*([a-zA-Z\-]+)([^()]*)\)")

def extract_section_facts_from_pddl(pddl_text: str):
  init_idx = pddl_text.find("(:init")
  goal_idx = pddl_text.find("(:goal")
  if init_idx == -1 or goal_idx == -1:
      raise ValueError("PDDL missing :init or :goal section.")

  init_text = pddl_text[init_idx:goal_idx]
  goal_text = pddl_text[goal_idx:]

  def _facts_from_section(sec: str):
      facts = []
      for m in _fact_line_re.finditer(sec):
          pred = m.group(1).strip()
          args = _norm_space(m.group(2)).split()

          if pred in {"clear", "on-table"} and len(args) == 1:
              facts.append((pred, args[0]))
          elif pred == "on" and len(args) == 2:
              facts.append((pred, args[0], args[1]))
          elif pred == "arm-empty" and len(args) == 0:
              facts.append((pred,))
      return facts

  return {
      "init_facts": _facts_from_section(init_text),
      "goal_facts": _facts_from_section(goal_text),
  }

In [ ]:
# Cell 23: Helper 6 - section-aware, order-insensitive diff
def diff_fact_sections(parsed_pddl_text: str, llm_pddl_text: str):
  parsed_secs = extract_section_facts_from_pddl(parsed_pddl_text)
  llm_secs = extract_section_facts_from_pddl(llm_pddl_text)

  parsed_init = set(parsed_secs["init_facts"])
  parsed_goal = set(parsed_secs["goal_facts"])
  llm_init = set(llm_secs["init_facts"])
  llm_goal = set(llm_secs["goal_facts"])

  return {
      "init_missing_in_llm": sorted(parsed_init - llm_init),
      "init_extra_in_llm": sorted(llm_init - parsed_init),
      "goal_missing_in_llm": sorted(parsed_goal - llm_goal),
      "goal_extra_in_llm": sorted(llm_goal - parsed_goal),
      "init_match_count": len(parsed_init & llm_init),
      "goal_match_count": len(parsed_goal & llm_goal),
      "init_total_parsed": len(parsed_init),
      "goal_total_parsed": len(parsed_goal),
      "init_total_llm": len(llm_init),
      "goal_total_llm": len(llm_goal),
      "perfect_match": (
          parsed_init == llm_init and
          parsed_goal == llm_goal
      ),
  }

In [ ]:
# Cell 24: Helper 7 - concise mismatch-only diff report text
def build_diff_report_text(problem_id: str, diff_result: dict):
  lines = []
  lines.append(f"Problem: {problem_id}")
  lines.append(f"Perfect match: {diff_result['perfect_match']}")
  lines.append("")

  lines.append(f"Init missing in LLM ({len(diff_result['init_missing_in_llm'])}):")
  for fact in diff_result["init_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Init extra in LLM ({len(diff_result['init_extra_in_llm'])}):")
  for fact in diff_result["init_extra_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal missing in LLM ({len(diff_result['goal_missing_in_llm'])}):")
  for fact in diff_result["goal_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal extra in LLM ({len(diff_result['goal_extra_in_llm'])}):")
  for fact in diff_result["goal_extra_in_llm"]:
      lines.append(f"- {fact}")

  return "\n".join(lines) + "\n"

In [ ]:
# Cell 25: Parser + diff run (single problem handle, persists outputs to pXXX/cand_00)
if not NL_PROBLEM_PATH.exists():
  raise FileNotFoundError(f"Missing NL description: {NL_PROBLEM_PATH}")
if not LLM_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing LLM-generated PDDL: {LLM_PDDL_PATH}")

nl_text = NL_PROBLEM_PATH.read_text()
llm_pddl_text = LLM_PDDL_PATH.read_text()

parsed_struct = parse_nl_problem_description(nl_text)
parsed_pddl_text = render_problem_pddl(PARSER_PROBLEM, parsed_struct)
diff_result = diff_fact_sections(parsed_pddl_text, llm_pddl_text)
diff_text = build_diff_report_text(PARSER_PROBLEM, diff_result)

PARSED_PDDL_OUT.write_text(parsed_pddl_text)
DIFF_JSON_OUT.write_text(json.dumps(diff_result, indent=2))
DIFF_TXT_OUT.write_text(diff_text)

print("Saved:")
print("-", PARSED_PDDL_OUT)
print("-", DIFF_JSON_OUT)
print("-", DIFF_TXT_OUT)
print("")
print("Mismatch summary:")
print(f"- init missing: {len(diff_result['init_missing_in_llm'])}")
print(f"- init extra:   {len(diff_result['init_extra_in_llm'])}")
print(f"- goal missing: {len(diff_result['goal_missing_in_llm'])}")
print(f"- goal extra:   {len(diff_result['goal_extra_in_llm'])}")
print(f"- perfect_match: {diff_result['perfect_match']}")